# Coming from R SVEMnet (or JMP Pro SVEM)

This notebook maps the R SVEMnet workflow onto `svemnet` for Python. The
Gaussian fitting mathematics are the same (anti-correlated fractional random
weights, validation-weighted wAIC selection, Kish admissibility guardrail,
ensemble averaging); the forward-selection engines validate against the R
functions `forward_aicc()` / `svem_forward()` exactly, replicate by
replicate, given the same random weights.

| R SVEMnet | svemnet (Python) |
|---|---|
| `SVEMnet(y ~ ..., data)` | `svemnet.svem("y ~ ...", df)` |
| `svem_forward(y ~ ..., data)` | `svemnet.svem("y ~ ...", df, method="forward")` |
| `forward_aicc(y ~ ..., data)` | `svemnet.forward_aicc("y ~ ...", df)` |
| `predict(fit, nd, se.fit=, interval=)` | `model.predict(nd, se_fit=, interval=)` |
| `coef(fit)`, `svem_nonzero(fit)` | `model.coef_table()` |
| `bigexp_terms(...)` | `svemnet.response_surface_formula(...)` |

In [ ]:
import numpy as np
import pandas as pd
import svemnet

# Factors: two continuous, one categorical (an R factor = pandas Categorical)
rng = np.random.default_rng(7)
n = 30
df = pd.DataFrame({
    "X1": rng.normal(size=n),
    "X2": rng.normal(size=n),
    "Line": pd.Categorical(rng.choice(["A", "B", "C"], n)),
})
df["y"] = (
    1 + 2 * df.X1 - 1.5 * df.X2
    + 2.0 * (df.Line == "B") - 1.0 * (df.Line == "C")
    + rng.normal(0, 0.4, n)
)

# R: spec <- bigexp_terms(y ~ X1 + X2 + Line, data, ...)
formula = svemnet.response_surface_formula(
    "y", continuous=["X1", "X2"], nominal=["Line"]
)
print(formula)

Categorical columns are treatment-coded like R factors, and in
forward-selection paths the whole contrast block (`Line[T.B]`, `Line[T.C]`)
enters or leaves **together**, matching the R behavior.

In [ ]:
fit = svemnet.svem(formula, df, method="forward", nBoot=200, seed=3)
print({k: round(v, 2) for k, v in fit.selection_frequencies.items()})
fit.coef_table().round(3)

In [ ]:
# Unseen factor levels: R SVEMnet returns NA with a warning; here prediction
# fails fast with an informative error instead.
bad = pd.DataFrame({
    "X1": [0.0], "X2": [0.0],
    "Line": pd.Categorical(["D"], categories=["A", "B", "C", "D"]),
})
try:
    fit.predict(bad)
except ValueError as err:
    print("ValueError:", str(err)[:120], "...")

## What stays in R (or JMP)

This package is intentionally minimal. For binomial responses, the
whole-model significance test, mixture-constrained random-search
optimization (`svem_score_random`), and Thompson-sampling batch design
(`svem_thompson_batch`), use the R package:
https://CRAN.R-project.org/package=SVEMnet — fitted models there and here
share conventions, so results are directly comparable.